In [ ]:
import os
import platform

try:
    import torch
except Exception:
    torch = None

print(f"Python version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")

if torch is None:
    print("PyTorch is not available yet. That is normal before installation.")
else:
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        total_vram_gb = props.total_memory / (1024 ** 3)
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Approx VRAM: {total_vram_gb:.1f} GB")
        if total_vram_gb < 14:
            print("Warning: this GPU is likely too small for a comfortable Qwen3-8B run, even in 4-bit mode.")
    else:
        print("No GPU detected. Switch the Colab runtime to GPU before continuing.")


Python version: 3.12.12
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Approx VRAM: 14.6 GB


In [ ]:
%pip install --quiet unsloth trl datasets accelerate peft bitsandbytes pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from google.colab import files
from transformers import TrainingArguments
from trl import SFTTrainer

print("All core imports succeeded.")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, switch to Runtime > Change runtime type > GPU.")

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)

print(f"GPU name: {gpu_name}")
print(f"Approx VRAM: {total_vram_gb:.1f} GB")

if total_vram_gb < 14:
    print("Warning: Qwen3-8B may be unstable on this runtime. The notebook can still be useful for learning, but memory errors are more likely.")
else:
    print("This looks reasonable for a 4-bit LoRA beginner run.")


All core imports succeeded.
CUDA available: True
GPU name: Tesla T4
Approx VRAM: 14.6 GB
This looks reasonable for a 4-bit LoRA beginner run.


In [ ]:
EXPECTED_KEYS = [
    "defect_type",
    "severity",
    "production_line",
    "part",
    "probable_cause",
    "next_action",
]
ALLOWED_SEVERITIES = {"low", "medium", "high", "critical"}

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

uploaded_name = next(iter(uploaded.keys()))
dataset_path = Path(uploaded_name)
print(f"Using uploaded file: {dataset_path}")

examples = []
errors = []

with dataset_path.open("r", encoding="utf-8") as handle:
    for line_number, raw_line in enumerate(handle, start=1):
        raw_line = raw_line.strip()
        if not raw_line:
            continue
        try:
            row = json.loads(raw_line)
        except json.JSONDecodeError as exc:
            errors.append(f"Line {line_number}: invalid JSON - {exc}")
            continue

        if not isinstance(row, dict):
            errors.append(f"Line {line_number}: each line must be a JSON object.")
            continue

        if "input" not in row:
            errors.append(f"Line {line_number}: missing 'input'.")
            continue
        if "output" not in row:
            errors.append(f"Line {line_number}: missing 'output'.")
            continue

        if not isinstance(row["input"], str):
            errors.append(f"Line {line_number}: 'input' must be a string.")
            continue
        if not isinstance(row["output"], dict):
            errors.append(f"Line {line_number}: 'output' must be a JSON object.")
            continue

        output_keys = list(row["output"].keys())
        if output_keys != EXPECTED_KEYS:
            errors.append(
                f"Line {line_number}: output keys must be exactly {EXPECTED_KEYS}, got {output_keys}."
            )
            continue

        severity = row["output"]["severity"]
        if severity not in ALLOWED_SEVERITIES:
            errors.append(
                f"Line {line_number}: invalid severity '{severity}'. Expected one of {sorted(ALLOWED_SEVERITIES)}."
            )
            continue

        examples.append(row)

print(f"Valid examples loaded: {len(examples)}")
print(f"Validation errors found: {len(errors)}")

if errors:
    print("\nFirst validation errors:")
    for err in errors[:10]:
        print("-", err)
    raise ValueError("Dataset validation failed. Fix the JSONL file and upload it again.")

preview_rows = []
for item in examples[:3]:
    preview_rows.append(
        {
            "input_preview": item["input"][:120] + ("..." if len(item["input"]) > 120 else ""),
            "severity": item["output"]["severity"],
            "defect_type": item["output"]["defect_type"],
            "line": item["output"]["production_line"],
        }
    )

pd.DataFrame(preview_rows)

Saving manufacturing_qa_500.jsonl to manufacturing_qa_500 (1).jsonl
Using uploaded file: manufacturing_qa_500 (1).jsonl
Valid examples loaded: 500
Validation errors found: 0


,input_preview,severity,defect_type,line
0,"So, line 1 — spotted an issue with the housing...",medium,assembly error,Line 1
1,Suspecting calibration drift on positioning sy...,high,misalignment,Line 3
2,Inspecting gear units from Line 2. Probably ba...,low,wrong label,Line 2


In [ ]:
TARGET_SCHEMA_EXAMPLE = {
    "defect_type": "misalignment",
    "severity": "high",
    "production_line": "Line 2",
    "part": "housing bracket",
    "probable_cause": "fixture drift after maintenance",
    "next_action": "stop the line, inspect current batch, and recalibrate the fixture"
}

SYSTEM_INSTRUCTION = (
    "You extract structured manufacturing incident data. "
    "Return JSON only. Do not add explanations, markdown, or extra text. "
    f"The JSON keys must be exactly: {EXPECTED_KEYS}."
)

print(json.dumps(TARGET_SCHEMA_EXAMPLE, indent=2))
print()
print(SYSTEM_INSTRUCTION)

{
  "defect_type": "misalignment",
  "severity": "high",
  "production_line": "Line 2",
  "part": "housing bracket",
  "probable_cause": "fixture drift after maintenance",
  "next_action": "stop the line, inspect current batch, and recalibrate the fixture"
}

You extract structured manufacturing incident data. Return JSON only. Do not add explanations, markdown, or extra text. The JSON keys must be exactly: ['defect_type', 'severity', 'production_line', 'part', 'probable_cause', 'next_action'].


In [ ]:
MODEL_NAME = "unsloth/Qwen3-8B"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("Base model loaded.")
print(f"EOS token: {repr(tokenizer.eos_token)}")

==((====))==  Unsloth 2026.3.8: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model loaded.
EOS token: '<|im_end|>'


In [ ]:
def build_inference_prompt(incident_text: str) -> str:
    return (
        f"{SYSTEM_INSTRUCTION}\n\n"
        "Incident report:\n"
        f"{incident_text.strip()}\n\n"
        "JSON:\n"
    )

def build_training_text(example: dict) -> str:
    target_json = json.dumps(example["output"], ensure_ascii=False)
    return build_inference_prompt(example["input"]) + target_json + tokenizer.eos_token

raw_dataset = Dataset.from_list(examples)
split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)

train_raw = split_dataset["train"]
test_raw = split_dataset["test"]

train_dataset = train_raw.map(lambda ex: {"text": build_training_text(ex)})
test_dataset = test_raw.map(lambda ex: {"text": build_training_text(ex)})

print(f"Train examples: {len(train_dataset)}")
print(f"Test examples: {len(test_dataset)}")

print("\nSample training text preview:\n")
print(train_dataset[0]["text"][:1200])

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Train examples: 450
Test examples: 50

Sample training text preview:

You extract structured manufacturing incident data. Return JSON only. Do not add explanations, markdown, or extra text. The JSON keys must be exactly: ['defect_type', 'severity', 'production_line', 'part', 'probable_cause', 'next_action'].

Incident report:
Hey, the last batch of lever arm from Assembly Line isn't right. Cause: worn conveyor belt surface. Issue: you can see lines across the surface. No prior issues with this lever arm batch.

JSON:
{"defect_type": "scratch", "next_action": "recall affected batch from warehouse", "part": "lever arm", "probable_cause": "worn conveyor belt surface", "production_line": "Assembly Line", "severity": "high"}<|im_end|>


In [ ]:
# FastLanguageModel.for_inference(model)

def generate_response(current_model, prompt: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = current_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            use_cache=False,
        )
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded.strip()

qualitative_examples = [test_raw[i] for i in range(min(3, len(test_raw)))]
metric_examples = [test_raw[i] for i in range(min(20, len(test_raw)))]

baseline_predictions = []
for item in metric_examples:
    baseline_predictions.append(generate_response(model, build_inference_prompt(item["input"])))

for idx, item in enumerate(qualitative_examples, start=1):
    baseline_output = generate_response(model, build_inference_prompt(item["input"]))
    print(f"\n--- Baseline example {idx} ---")
    print("INPUT:\n", item["input"])
    print("\nBASE MODEL OUTPUT:\n", baseline_output)
    print("\nEXPECTED OUTPUT:\n", json.dumps(item["output"], indent=2))


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


--- Baseline example 1 ---
INPUT:
 during the night shift run on Line 1, the bracket had an issue. Looks like it wobbles when you touch it. My guess is cross-threaded bolt.

BASE MODEL OUTPUT:
 {
  "defect_type": "wobble",
  "severity": "medium",
  "production_line": "Line 1",
  "part": "bracket",
  "probable_cause": "cross-threaded bolt",
  "next_action": "inspect all bolts on Line 1"
}
Okay, let's tackle this. The user wants me to extract structured manufacturing incident data into a specific JSON format. The keys must be exactly ['defect_type', 'severity', 'production_line', 'part', 'probable_cause', 'next_action']. 

First, I'll read the incident report carefully. The report says: "during the night shift run on Line 1, the bracket had an issue. Looks like it wobbles when you touch it. My guess is cross-threaded bolt."

So, breaking it down:

- The defect type is mentioned as "wobbles when you touch it," so "wobble" makes sense for defect_type.
- Severity is "medium" because it's a

Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Baseline example 2 ---
INPUT:
 Line 1, control module batch. Not acceptable — there's moisture on the outside. Seems to be improper gasket seating.

BASE MODEL OUTPUT:
 {
  "defect_type": "moisture contamination",
  "severity": "medium",
  "production_line": "Line 1",
  "part": "control module",
  "probable_cause": "improper gasket seating",
  "next_action": "inspect all gaskets on Line 1 and replace as necessary"
}
Okay, let's tackle this. The user wants me to extract structured manufacturing incident data into a specific JSON format. First, I need to read the incident report carefully. The report mentions Line 1, control module batch. The issue is moisture on the outside, which is not acceptable. The probable cause seems to be improper gasket seating.

So, the JSON keys required are defect_type, severity, production_line, part, probable_cause, and next_action. Let's map each part. The defect type here is moisture contamination. Severity is mentioned as "Not acceptable" but the e

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

training_args = TrainingArguments(
    output_dir="qwen3_8b_manufacturing_outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

train_result = trainer.train()
train_result

Unsloth 2026.3.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/450 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)


Step,Training Loss
5,2.865270
10,2.099109
15,1.461795
20,1.084212
25,1.001081
30,0.911950
35,0.802288
40,0.774390
45,0.737211
50,0.649530


TrainOutput(global_step=60, training_loss=1.1438941597938537, metrics={'train_runtime': 223.8913, 'train_samples_per_second': 2.144, 'train_steps_per_second': 0.268, 'total_flos': 3447529016401920.0, 'train_loss': 1.1438941597938537, 'epoch': 1.0533333333333332})

In [ ]:
FastLanguageModel.for_inference(model)

def extract_first_json_object(text: str):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    candidate = text[start:end + 1]
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return None

def score_predictions(reference_examples, predictions):
    summary = {
        "total": len(reference_examples),
        "json_valid": 0,
        "all_fields_present": 0,
        "severity_exact": 0,
        "production_line_exact": 0,
    }

    for example, prediction in zip(reference_examples, predictions):
        parsed = extract_first_json_object(prediction)
        if parsed is None:
            continue

        summary["json_valid"] += 1
        if list(parsed.keys()) == EXPECTED_KEYS:
            summary["all_fields_present"] += 1
        if parsed.get("severity") == example["output"]["severity"]:
            summary["severity_exact"] += 1
        if parsed.get("production_line") == example["output"]["production_line"]:
            summary["production_line_exact"] += 1

    return {
        key: value if key == "total" else round(value / max(summary["total"], 1), 3)
        for key, value in summary.items()
    }

fine_tuned_predictions = []
for item in metric_examples:
    fine_tuned_predictions.append(generate_response(model, build_inference_prompt(item["input"])))

baseline_scores = score_predictions(metric_examples, baseline_predictions)
fine_tuned_scores = score_predictions(metric_examples, fine_tuned_predictions)

comparison_df = pd.DataFrame([baseline_scores, fine_tuned_scores], index=["base_model", "fine_tuned"])
display(comparison_df)

for idx, item in enumerate(qualitative_examples, start=1):
    fine_tuned_output = generate_response(model, build_inference_prompt(item["input"]))
    print(f"\n--- Fine-tuned example {idx} ---")
    print("INPUT:\n", item["input"])
    print("\nFINE-TUNED OUTPUT:\n", fine_tuned_output)
    print("\nEXPECTED OUTPUT:\n", json.dumps(item["output"], indent=2))


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

,total,json_valid,all_fields_present,severity_exact,production_line_exact
base_model,20,0.35,0.35,0.1,0.3
fine_tuned,20,1.00,0.00,0.3,1.0


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2


--- Fine-tuned example 1 ---
INPUT:
 during the night shift run on Line 1, the bracket had an issue. Looks like it wobbles when you touch it. My guess is cross-threaded bolt.

FINE-TUNED OUTPUT:
 {"defect_type": "loose fastener", "next_action": "issue non-conformance report", "part": "bracket", "probable_cause": "cross-threaded bolt", "production_line": "Line 1", "severity": "high"}

EXPECTED OUTPUT:
 {
  "defect_type": "loose fastener",
  "next_action": "adjust process parameters and re-verify",
  "part": "bracket",
  "probable_cause": "cross-threaded bolt",
  "production_line": "Line 1",
  "severity": "medium"
}


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Fine-tuned example 2 ---
INPUT:
 Line 1, control module batch. Not acceptable — there's moisture on the outside. Seems to be improper gasket seating.

FINE-TUNED OUTPUT:
 {"defect_type": "leak", "next_action": "perform root cause analysis", "part": "control module", "probable_cause": "improper gasket seating", "production_line": "Line 1", "severity": "medium"}

EXPECTED OUTPUT:
 {
  "defect_type": "leak",
  "next_action": "mark unit for rework at end of shift",
  "part": "control module",
  "probable_cause": "improper gasket seating",
  "production_line": "Line 1",
  "severity": "low"
}

--- Fine-tuned example 3 ---
INPUT:
 Small update — during the second shift run on Line 3, the housing had an issue. Traced back to leftover labels from previous batch run. Noticed somebody put the wrong tag on it. Double-checked with instruments.

FINE-TUNED OUTPUT:
 {"defect_type": "wrong label", "next_action": "issue non-conformance report", "part": "housing", "probable_cause": "leftover labels

In [ ]:
import shutil

OUTPUT_DIR = "qwen3_8b_manufacturing_lora"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

archive_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print(f"Saved adapters and tokenizer to: {OUTPUT_DIR}")
print(f"Zip archive created at: {archive_path}")

print("If you want to download the zip manually, use the Files pane in Colab.")

Saved adapters and tokenizer to: qwen3_8b_manufacturing_lora
Zip archive created at: /content/qwen3_8b_manufacturing_lora.zip
If you want to download the zip manually, use the Files pane in Colab.
